In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

# Load data - PATH UPDATED FOR YOUR FOLDER STRUCTURE
train = pd.read_csv('../data/dataset/train.csv')
test = pd.read_csv('../data/dataset/test.csv')
sample_sub = pd.read_csv('../data/dataset/sample_submission.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (77299, 11)
Test shape: (41778, 10)


In [2]:
def preprocess_data(df):
    # 1. Time Parsing (Timestamp is "hour:minute")
    df[['hour', 'minute']] = df['timestamp'].str.split(':', expand=True).astype(int)
    df['total_minutes'] = df['hour'] * 60 + df['minute']
    
    # Cyclic encoding so model knows 23:45 is close to 00:00
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['min_sin'] = np.sin(2 * np.pi * df['total_minutes'] / 1440)
    df['min_cos'] = np.cos(2 * np.pi * df['total_minutes'] / 1440)
    
    # 2. Smart Imputation
    # Temperature: fill by geohash median (location-based weather)
    df['Temperature'] = df.groupby('geohash')['Temperature'].transform(lambda x: x.fillna(x.median()))
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median()) # fallback
    
    # Categoricals: Fill with "Unknown"
    df['Weather'] = df['Weather'].fillna('Unknown')
    df['RoadType'] = df['RoadType'].fillna('Unknown')
    
    return df

train = preprocess_data(train)
test = preprocess_data(test)

In [3]:
# pip install pygeohash (run this in your terminal once)
import pygeohash as pgh

def decode_geohash(df):
    # Extract hierarchical prefixes (Neighborhoods)
    df['geo_p2'] = df['geohash'].str[:2]
    df['geo_p3'] = df['geohash'].str[:3]
    df['geo_p4'] = df['geohash'].str[:4]
    
    # Decode to exact Lat/Lon
    lats, lons = [], []
    for gh in df['geohash']:
        try:
            lat, lon = pgh.decode(gh)
            lats.append(lat); lons.append(lon)
        except:
            lats.append(np.nan); lons.append(np.nan)
    df['latitude'] = lats
    df['longitude'] = lons
    return df

train = decode_geohash(train)
test = decode_geohash(test)

In [ ]:
# 1. Combine Train and Test
test['demand'] = np.nan
combined = pd.concat([train, test], axis=0).reset_index(drop=True)

# 2. TARGET TRANSFORMATION (Do this before lags)
combined['demand'] = np.log1p(combined['demand'])

# 3. SAFE 1-DAY LAG (Merge instead of Shift to prevent missing timestamp issues)
# Create a temporary dataframe with yesterday's demand
yesterday_df = combined[['geohash', 'day', 'hour', 'minute', 'demand']].copy()
yesterday_df['day'] = yesterday_df['day'] + 1 # Shift day forward by 1
yesterday_df.rename(columns={'demand': 'demand_1day_ago'}, inplace=True)

# Merge it back into combined
combined = combined.merge(yesterday_df, on=['geohash', 'day', 'hour', 'minute'], how='left')

# Drop the broken short-term lags if they still exist
for col in ['demand_lag_1', 'demand_lag_4', 'demand_rolling_4']:
    if col in combined.columns:
        combined.drop(col, axis=1, inplace=True)

# 4. Split back into Train and Test
train = combined[combined['demand'].notna()].copy()
test = combined[combined['demand'].isna()].drop('demand', axis=1).copy()

# 5. Safe Target Encoding (Expanding Mean - No Future Leakage!)
def calc_expanding_mean(df, group_cols, target):
    df[f'{"".join(group_cols)}_target_enc'] = df.groupby(group_cols)[target].transform(
        lambda x: x.expanding().mean().shift(1)
    )
    global_mean = df[target].mean()
    df[f'{"".join(group_cols)}_target_enc'] = df[f'{"".join(group_cols)}_target_enc'].fillna(global_mean)
    return df

train = calc_expanding_mean(train, ['geohash', 'hour'], 'demand')
train = calc_expanding_mean(train, ['Weather', 'RoadType'], 'demand')

# For the test set, use the full train averages
agg_geo_hour = train.groupby(['geohash', 'hour'])['demand'].mean().reset_index()
agg_geo_hour.columns = ['geohash', 'hour', 'geohashhour_target_enc']
test = test.merge(agg_geo_hour, on=['geohash', 'hour'], how='left')

agg_weather_road = train.groupby(['Weather', 'RoadType'])['demand'].mean().reset_index()
agg_weather_road.columns = ['Weather', 'RoadType', 'WeatherRoadType_target_enc']
test = test.merge(agg_weather_road, on=['Weather', 'RoadType'], how='left')

test['geohashhour_target_enc'] = test['geohashhour_target_enc'].fillna(train['demand'].mean())
test['WeatherRoadType_target_enc'] = test['WeatherRoadType_target_enc'].fillna(train['demand'].mean())

print("Lag features and aggregations created safely!")

In [5]:
cat_cols = ['geohash', 'geo_p2', 'geo_p3', 'geo_p4', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    combined_str = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined_str)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))
    label_encoders[col] = le

# Drop old leaky cols if they somehow survived
for col in ['geo_hour_mean', 'geo_hour_std', 'weather_road_mean']:
    if col in train.columns: train.drop(col, axis=1, inplace=True)
    if col in test.columns: test.drop(col, axis=1, inplace=True)

# Feature Interactions
for df in [train, test]:
    df['lanes_x_hour'] = df['NumberofLanes'] * df['hour']
    df['lanes_x_weather'] = df['NumberofLanes'] * df['Weather']

drop_cols = ['Index', 'timestamp', 'demand_original', 'demand']
features = [c for c in train.columns if c not in drop_cols]

X_train = train[features]
y_train = train['demand']
X_test = test[features]

print("Training features:", len(features))

Training features: 28


In [6]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score

# TimeSeriesSplit respects time order! No more peeking into the future.
tscv = TimeSeriesSplit(n_splits=5)

lgb_preds = np.zeros(len(X_test))
xgb_preds = np.zeros(len(X_test))
cat_preds = np.zeros(len(X_test))

lgb_scores, xgb_scores, cat_scores = [], [], []

for fold, (trn_idx, val_idx) in enumerate(tscv.split(X_train)):
    X_tr, X_val = X_train.iloc[trn_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[trn_idx], y_train.iloc[val_idx]
    
    # --- LightGBM ---
    lgb_model = lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.05, max_depth=8, 
        num_leaves=60, subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbose=-1
    )
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
    lgb_preds += lgb_model.predict(X_test) / tscv.n_splits
    
    val_pred_lgb = np.clip(np.expm1(lgb_model.predict(X_val)), 0, None)
    lgb_score = r2_score(np.expm1(y_val), val_pred_lgb)
    lgb_scores.append(lgb_score)
    
    # --- XGBoost ---
    xgb_model = xgb.XGBRegressor(
        n_estimators=2000, learning_rate=0.05, max_depth=7,
        subsample=0.8, colsample_bytree=0.8, random_state=42, 
        tree_method='hist', early_stopping_rounds=50
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    xgb_preds += xgb_model.predict(X_test) / tscv.n_splits
    
    val_pred_xgb = np.clip(np.expm1(xgb_model.predict(X_val)), 0, None)
    xgb_score = r2_score(np.expm1(y_val), val_pred_xgb)
    xgb_scores.append(xgb_score)
    
    # --- CatBoost ---
    cat_model = CatBoostRegressor(
        iterations=2000, learning_rate=0.05, depth=8,
        random_seed=42, verbose=False
    )
    cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=False)
    cat_preds += cat_model.predict(X_test) / tscv.n_splits
    
    val_pred_cat = np.clip(np.expm1(cat_model.predict(X_val)), 0, None)
    cat_score = r2_score(np.expm1(y_val), val_pred_cat)
    cat_scores.append(cat_score)
    
    print(f"Fold {fold+1} Complete | LGB R2: {lgb_score:.4f} | XGB R2: {xgb_score:.4f} | CAT R2: {cat_score:.4f}")

print("\n========================================")
print(f"Average LGB R2: {np.mean(lgb_scores):.4f}")
print(f"Average XGB R2: {np.mean(xgb_scores):.4f}")
print(f"Average CAT R2: {np.mean(cat_scores):.4f}")
print("========================================")

Fold 1 Complete | LGB R2: 0.9469 | XGB R2: 0.9393 | CAT R2: 0.9324
Fold 2 Complete | LGB R2: 0.9114 | XGB R2: 0.9086 | CAT R2: 0.9137
Fold 3 Complete | LGB R2: 0.9637 | XGB R2: 0.9376 | CAT R2: 0.9557
Fold 4 Complete | LGB R2: 0.9728 | XGB R2: 0.9603 | CAT R2: 0.9710
Fold 5 Complete | LGB R2: 0.9292 | XGB R2: 0.9293 | CAT R2: 0.9304

Average LGB R2: 0.9448
Average XGB R2: 0.9350
Average CAT R2: 0.9406


In [7]:
# Rock-solid optimized weighted blend
final_preds_log = (0.50 * lgb_preds) + (0.30 * xgb_preds) + (0.20 * cat_preds)

# Inverse Transform (Undo the log1p)
final_preds = np.expm1(final_preds_log)

# Clip predictions (Demand cannot be negative!)
final_preds = np.clip(final_preds, 0, None)

print("Predictions ready. Min:", final_preds.min(), "Max:", final_preds.max())

Predictions ready. Min: 0.00660870441303317 Max: 0.8846701850487446


In [ ]:
import os
os.makedirs('../submissions', exist_ok=True)

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': final_preds
})

# CRITICAL: Sort by Index to match the original test.csv row order!
submission = submission.sort_values('Index').reset_index(drop=True)

submission.to_csv('../submissions/submission.csv', index=False)
print("Submission saved! Shape:", submission.shape)

Submission saved! Shape: (41778, 2)
